# 14 — Heuristic Defenses & Input Pre-Processing — Lab

## Description

This lab exercises feature-squeezing pipelines and defense evaluation protocols. You will implement a three-stage pre-processing defense (bit-depth reduction, median filtering, JPEG compression), generate adversarial examples using FGSM and PGD, and measure the defense's effectiveness across clean and adversarial inputs. By the end, you will have empirical evidence of how pre-processing defenses trade clean accuracy against adversarial robustness.

## Objectives

- Implement a feature-squeezing filter pipeline combining bit-depth reduction and spatial smoothing in PyTorch.
- Explain how JPEG compression removes high-frequency adversarial perturbation energy via DCT and quantization.
- Measure defense efficacy (clean accuracy drop, adversarial accuracy) for each pre-processing method against FGSM and PGD at matched epsilon.

## Prerequisites

You should have completed the lecture content for m14. You should also be familiar with FGSM and PGD attacks from m08 (white-box evasion attacks), and understand how to generate and evaluate adversarial examples in PyTorch.

## Required Software/Packages

- PyTorch (torch, torchvision) version 2.0 or later
- NumPy version 1.20 or later
- Pillow (PIL) version 8.0 or later for image processing
- SciPy version 1.7 or later for median filtering
- Matplotlib version 3.3 or later for visualization
- Adversarial Robustness Toolbox (`adversarial-robustness-toolbox`) version 1.17 or later, for the FGSM/PGD attacks and the pre-processing defenses
- `kornia`, ART's dependency for the PyTorch spatial-smoothing (median filter) defense

## Environment and Setup Instructions

Run the cells below to set up the environment and train a baseline ResNet-18 model on CIFAR-10. This setup is prerequisite scaffolding and is not counted toward the exercise time budget. The training will take approximately 5-10 minutes.

In [ ]:
# This lab uses the Adversarial Robustness Toolbox (ART) for both the FGSM/PGD attacks
# and the three pre-processing defenses (feature squeezing, spatial smoothing, JPEG compression).
# kornia is ART's dependency for the PyTorch spatial-smoothing (median filter) defense.
!pip install adversarial-robustness-toolbox kornia

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
import numpy as np
from PIL import Image
import io
from scipy.ndimage import median_filter
import matplotlib.pyplot as plt

# ART components used in this lab:
# - PyTorchClassifier wraps a PyTorch model in ART's estimator interface.
# - FastGradientMethod / ProjectedGradientDescent are the FGSM and PGD evasion attacks (m08/m09).
# - FeatureSqueezing, SpatialSmoothingPyTorch, and JpegCompression are the three pre-processing
#   defenses you will wire up below, by calling them by name.
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent
from art.defences.preprocessor import FeatureSqueezing, SpatialSmoothingPyTorch, JpegCompression

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# SETUP: Train a ResNet-18 baseline model on CIFAR-10
# This is not part of the 30-minute exercise time budget; it is prerequisite scaffolding.

print("Preparing CIFAR-10 data for training...")
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, transform=transform_train, download=True)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

test_dataset = datasets.CIFAR10(root='./data', train=False, transform=transform_test, download=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Training set size: {len(train_dataset)}, Test set size: {len(test_dataset)}")

# Define and train ResNet-18
model = models.resnet18(weights=None)
model.fc = nn.Linear(512, 10)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=3)

print("\nTraining ResNet-18 on CIFAR-10 (3 epochs)...")
for epoch in range(3):
    model.train()
    total_loss = 0.0
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if (batch_idx + 1) % 100 == 0:
            print(f"  Epoch {epoch+1}/3, Batch {batch_idx+1}/{len(train_loader)}, Loss: {loss.item():.4f}")
    
    scheduler.step()
    
    # Evaluate on test set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    test_accuracy = 100.0 * correct / total
    print(f"  Epoch {epoch+1}/3 — Test Accuracy: {test_accuracy:.2f}%")

# Save the trained model
torch.save(model.state_dict(), './pretrained_resnet18_cifar10.pth')
print("\nTraining complete. Model saved to ./pretrained_resnet18_cifar10.pth")

## Background

Pre-processing defenses operate before the classifier sees the input. They aim to remove or reduce adversarial perturbations, which often concentrate in high-frequency components or exist at sub-pixel scales. This lab focuses on three heuristic defenses: feature squeezing (quantization plus smoothing), JPEG compression (frequency-domain filtering), and evaluating their joint effect. You will also use FGSM and PGD attacks, built on the Adversarial Robustness Toolbox (ART), to test whether these defenses truly reduce adversarial accuracy.

The FGSM attack and the bit-depth-reduction defense are implemented for you below as worked examples, using ART's `FastGradientMethod` and `FeatureSqueezing`. You will then implement PGD (which you have already built from scratch in m08/m09, here called by name through ART) and the two remaining defenses, `median_filter_defense` and `jpeg_compression_defense`, by calling ART's `SpatialSmoothingPyTorch` and `JpegCompression`. Each of those three TODOs links to ART's documentation and source code for the class involved, so you know exactly what to look up if you get stuck.

## Exercise Instructions

1. Load the trained model and prepare the test dataset (first 500 examples) (3 min).
2. Read the FGSM attack function: it is implemented for you using ART's `FastGradientMethod`, at epsilon = 8/255. This is the same attack you built from scratch in m08; here you call it by name (3 min).
3. Implement a PGD attack function with step size alpha = 2/255, 40 iterations, and epsilon = 8/255, using ART's `ProjectedGradientDescent` the same way the FGSM example uses `FastGradientMethod` (5 min).
4. Read the bit-depth-reduction (feature-squeezing) defense: it is implemented for you using ART's `FeatureSqueezing`, quantizing to 3 bits per channel (3 min).
5. Implement a function that applies a 3 by 3 median filter to an image tensor, using ART's `SpatialSmoothingPyTorch` (5 min).
6. Implement a function that applies JPEG compression at quality factor Q = 75, using ART's `JpegCompression` (5 min).
7. Evaluate classifier accuracy on three defense configurations: (1) no defense, (2) bit-depth plus median, (3) full pipeline. Test on clean, FGSM, and PGD examples, and display results as a 3 by 3 table (5 min).

## Step 1: Load Trained Model and Prepare Test Data (3 min)

In [ ]:
# Load the trained model
model = models.resnet18(weights=None)
model.fc = nn.Linear(512, 10)
model.load_state_dict(torch.load('./pretrained_resnet18_cifar10.pth'))
model.to(device)
model.eval()

print("Model loaded and set to evaluation mode.")

# Load CIFAR-10 test dataset (first 500 examples)
transform = transforms.Compose([transforms.ToTensor()])
test_dataset = datasets.CIFAR10(root='./data', train=False, transform=transform, download=False)
test_subset = torch.utils.data.Subset(test_dataset, list(range(500)))
test_loader = torch.utils.data.DataLoader(test_subset, batch_size=50, shuffle=False)

# Collect all test images and labels
all_images = []
all_labels = []
for images, labels in test_loader:
    all_images.append(images)
    all_labels.append(labels)

X_test = torch.cat(all_images, dim=0)
y_test = torch.cat(all_labels, dim=0)

print(f"Test set shape: {X_test.shape}, labels shape: {y_test.shape}")

# Verify baseline accuracy
with torch.no_grad():
    X_test_device = X_test.to(device)
    outputs = model(X_test_device)
    _, preds = torch.max(outputs, 1)
    baseline_accuracy = (preds.cpu() == y_test).sum().item() / len(y_test) * 100
    print(f"Baseline accuracy on test set (500 examples): {baseline_accuracy:.2f}%")

**Expected Output:** The baseline accuracy should be approximately 70 to 75 percent on the 500-example test set, confirming the model is trained and functional.

## Step 2: FGSM Attack, Worked Example (3 min)

Read the FGSM attack function below. It is implemented for you using ART's `FastGradientMethod`, generating adversarial examples by taking one step in the direction of the gradient of the loss with respect to the input, exactly like the attack you built from scratch in m08.

In [ ]:
def fgsm_attack(model, images, labels, epsilon=8/255):
    """
    Generate FGSM adversarial examples using ART's FastGradientMethod.
    Args: model (trained classifier), images (tensor), labels (tensor), epsilon (perturbation budget)
    Returns: adv_images (adversarial examples clipped to [0, 1])

    Worked example: you implemented FGSM by hand in m08 (one step in the sign-of-the-gradient
    direction). Here the same attack is produced by wrapping the model in ART's
    PyTorchClassifier and calling FastGradientMethod by name, which is the pattern you will
    reuse for PGD in the next step.
    """
    classifier = PyTorchClassifier(
        model=model,
        loss=nn.CrossEntropyLoss(),
        input_shape=tuple(images.shape[1:]),
        nb_classes=10,
        clip_values=(0.0, 1.0),
        device_type="gpu" if images.device.type == "cuda" else "cpu",
    )
    attack = FastGradientMethod(estimator=classifier, eps=epsilon, norm=np.inf)

    # ART attacks take and return NumPy arrays, with one-hot labels.
    x_np = images.detach().cpu().numpy()
    y_np = F.one_hot(labels, num_classes=10).float().cpu().numpy()
    x_adv_np = attack.generate(x=x_np, y=y_np)

    return torch.from_numpy(x_adv_np).float().to(images.device).clamp(0, 1)

# Test FGSM on a small batch
test_batch = X_test[:10].clone().to(device)
test_labels = y_test[:10].to(device)

fgsm_examples = fgsm_attack(model, test_batch, test_labels, epsilon=8/255)
print(f"FGSM examples shape: {fgsm_examples.shape}")
print(f"FGSM perturbation range: [{(fgsm_examples - test_batch).min():.4f}, {(fgsm_examples - test_batch).max():.4f}]")

**Expected Output:** FGSM examples should have perturbation range approximately in [-8/255, 8/255], or around [-0.0314, 0.0314]. Shape should be (10, 3, 32, 32).

## Step 3: Implement PGD Attack (5 min)

You already built PGD from scratch in m08/m09 (projected gradient descent with a random start, per-step projection, and clipping). Here, implement it by calling ART's `ProjectedGradientDescent` with the same three-line pattern the `fgsm_attack` worked example above uses for `FastGradientMethod`: wrap `model` in a `PyTorchClassifier`, build the attack object, then call `.generate(x=..., y=...)` on NumPy arrays and convert the result back to a tensor.

In [ ]:
def pgd_attack(model, images, labels, epsilon=8/255, alpha=2/255, num_iter=40):
    """
    Generate PGD adversarial examples.
    Args: model, images, labels, epsilon (perturbation budget), alpha (step size), num_iter (iterations)
    Returns: adv_images (adversarial examples within epsilon ball, clipped to [0, 1])
    """
    # TODO: Implement PGD using ART's ProjectedGradientDescent (same pattern as fgsm_attack above)
    # 1. Wrap `model` in a PyTorchClassifier, exactly as in fgsm_attack:
    #    - loss=nn.CrossEntropyLoss(), input_shape=tuple(images.shape[1:]), nb_classes=10,
    #      clip_values=(0.0, 1.0), device_type matching images.device.
    # 2. Build the attack: ProjectedGradientDescent(estimator=classifier, eps=epsilon,
    #    eps_step=alpha, max_iter=num_iter, norm=np.inf). ART's eps_step is this attack's alpha,
    #    and max_iter is this attack's num_iter.
    # 3. Convert `images` to a NumPy array and `labels` to one-hot (F.one_hot(labels,
    #    num_classes=10).float()), exactly as in fgsm_attack.
    # 4. Call attack.generate(x=..., y=...), then convert the result back to a tensor on
    #    images.device with torch.from_numpy(...).float().clamp(0, 1).
    pass

# Test PGD on a small batch
pgd_examples = pgd_attack(model, test_batch, test_labels, epsilon=8/255, alpha=2/255, num_iter=40)
print(f"PGD examples shape: {pgd_examples.shape}")
print(f"PGD perturbation range: [{(pgd_examples - test_batch).min():.4f}, {(pgd_examples - test_batch).max():.4f}]")

**Expected Output:** PGD examples should have perturbations within [-8/255, 8/255]. The attack should produce more effective adversarial examples than FGSM due to multiple optimization steps.

## Step 4: Bit-Depth Reduction, Worked Example (3 min)

Read the bit-depth-reduction function below. It is implemented for you as a worked example using ART's `FeatureSqueezing` preprocessing defense, which quantizes each pixel to a fixed number of bits per channel.

- Documentation: https://adversarial-robustness-toolbox.readthedocs.io/en/latest/modules/defences/preprocessor.html#feature-squeezing
- Source code: https://github.com/Trusted-AI/adversarial-robustness-toolbox/blob/main/art/defences/preprocessor/feature_squeezing.py

In [ ]:
def bit_depth_reduction(images, bits=3):
    """
    Reduce bit depth by quantizing to the specified number of bits per channel.
    Args: images (tensor, values in [0, 1]), bits (number of bits per channel)
    Returns: quantized images (tensor, values in [0, 1])

    Worked example: uses ART's FeatureSqueezing preprocessing defense, which quantizes each
    pixel to 2**bits - 1 levels on the [0, 1] scale. See the Step 4 markdown cell above for
    ART's documentation and source-code links.
    """
    squeeze = FeatureSqueezing(clip_values=(0.0, 1.0), bit_depth=bits)

    # ART preprocessing defenses take and return NumPy arrays.
    x_np = images.detach().cpu().numpy()
    x_squeezed_np, _ = squeeze(x_np)

    return torch.from_numpy(x_squeezed_np).float().to(images.device).clamp(0, 1)

# Test bit-depth reduction
reduced = bit_depth_reduction(test_batch, bits=3)
print(f"Original range: [{test_batch.min():.4f}, {test_batch.max():.4f}]")
print(f"3-bit quantized range: [{reduced.min():.4f}, {reduced.max():.4f}]")
print(f"Number of unique values per channel after 3-bit: {len(torch.unique(reduced[0, 0]))}")
print(f"Expected: 8 unique values (2^3)")

**Expected Output:** The quantized image should have approximately 8 unique values per channel (2^3). Range should remain [0, 1].

## Step 5: Implement Median Filter Defense (5 min)

Complete the median filter function below using ART's `SpatialSmoothingPyTorch` preprocessing defense (a median filter over a sliding window, implemented in PyTorch/kornia).

- Documentation: https://adversarial-robustness-toolbox.readthedocs.io/en/latest/modules/defences/preprocessor.html#spatial-smoothing-pytorch
- Source code: https://github.com/Trusted-AI/adversarial-robustness-toolbox/blob/main/art/defences/preprocessor/spatial_smoothing_pytorch.py

In [ ]:
def median_filter_defense(images, kernel_size=3):
    """
    Apply median filter to each channel of the batch.
    Args: images (tensor, shape [batch, channels, height, width]), kernel_size (filter kernel size)
    Returns: filtered images (tensor, clamped to [0, 1])
    """
    # TODO: Implement using ART's SpatialSmoothingPyTorch (see the Step 5 markdown cell above
    # for the documentation and source-code links). Follow the same pattern as
    # bit_depth_reduction above:
    # 1. Build the defense object: SpatialSmoothingPyTorch(window_size=kernel_size,
    #    channels_first=True, clip_values=(0.0, 1.0)). `channels_first=True` matches our
    #    tensor layout [batch, channels, height, width].
    # 2. Convert `images` to a NumPy array with images.detach().cpu().numpy().
    # 3. Call the defense object on the array, e.g. x_smoothed_np, _ = smooth(x_np), the same
    #    way FeatureSqueezing is called in bit_depth_reduction above.
    # 4. Convert the result back to a tensor on images.device with
    #    torch.from_numpy(...).float().clamp(0, 1).
    pass

# Test median filter
filtered = median_filter_defense(test_batch, kernel_size=3)
print(f"Median filter applied. Shape: {filtered.shape}")
print(f"Difference from original (MSE): {F.mse_loss(test_batch, filtered).item():.6f}")
print(f"Expected: small MSE (< 0.01 for clean images)")

**Expected Output:** Filtered image should have low MSE difference from original (< 0.01) for clean images. Shape should remain unchanged.

## Step 6: Implement JPEG Compression Defense (5 min)

Complete the JPEG compression function below using ART's `JpegCompression` preprocessing defense, which applies JPEG compression (DCT plus quantization) to each image.

- Documentation: https://adversarial-robustness-toolbox.readthedocs.io/en/latest/modules/defences/preprocessor.html#jpeg-compression
- Source code: https://github.com/Trusted-AI/adversarial-robustness-toolbox/blob/main/art/defences/preprocessor/jpeg_compression.py

In [ ]:
def jpeg_compression_defense(images, quality=75):
    """
    Apply JPEG compression to each image in the batch.
    Args: images (tensor, shape [batch, 3, height, width]), quality (JPEG quality factor 1-100)
    Returns: JPEG-compressed images (tensor, clamped to [0, 1])
    """
    # TODO: Implement using ART's JpegCompression (see the Step 6 markdown cell above for the
    # documentation and source-code links). Follow the same pattern as bit_depth_reduction
    # and median_filter_defense above:
    # 1. Build the defense object: JpegCompression(clip_values=(0.0, 1.0), quality=quality,
    #    channels_first=True). `channels_first=True` matches our tensor layout
    #    [batch, channels, height, width].
    # 2. Convert `images` to a NumPy array with images.detach().cpu().numpy().
    # 3. Call the defense object on the array, e.g. x_jpeg_np, _ = jpeg(x_np).
    # 4. Convert the result back to a tensor on images.device with
    #    torch.from_numpy(...).float().clamp(0, 1).
    pass

# Test JPEG compression
jpeg_compressed = jpeg_compression_defense(test_batch[:5], quality=75)
print(f"JPEG compressed. Shape: {jpeg_compressed.shape}")
print(f"Difference from original (MSE): {F.mse_loss(test_batch[:5], jpeg_compressed).item():.6f}")
print(f"Expected: small MSE (< 0.05 for clean images at Q=75)")

**Expected Output:** JPEG-compressed images should have MSE difference < 0.05 for Q=75. Visual quality should be preserved while high-frequency components are attenuated.

## Step 7: Evaluate All Defense Configurations (5 min)

Evaluate all three defense configurations against the trained model on clean, FGSM, and PGD examples, and display results as a 3x3 table (rows: defense variants, columns: input types).

In [ ]:
def evaluate_defense(model, X, y, defense_fn=None):
    """
    Evaluate classifier accuracy on given examples, optionally applying defense.
    Args: model, X (input images), y (labels), defense_fn (optional defense function)
    Returns: accuracy (percentage, 0-100)
    """
    model.eval()
    device_cur = next(model.parameters()).device
    
    batch_size = 50
    total_correct = 0
    total_samples = 0
    
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            batch_x = X[i:i+batch_size].to(device_cur)
            batch_y = y[i:i+batch_size].to(device_cur)
            
            if defense_fn is not None:
                batch_x = defense_fn(batch_x)
            
            outputs = model(batch_x)
            _, preds = torch.max(outputs, 1)
            total_correct += (preds == batch_y).sum().item()
            total_samples += batch_y.size(0)
    
    accuracy = 100.0 * total_correct / total_samples
    return accuracy

# Generate adversarial examples for full test set
print("Generating FGSM and PGD adversarial examples for 500 test samples...")
X_fgsm = []
X_pgd = []
batch_size = 50

for i in range(0, len(X_test), batch_size):
    batch_x = X_test[i:i+batch_size].clone().to(device)
    batch_y = y_test[i:i+batch_size].to(device)
    
    fgsm_batch = fgsm_attack(model, batch_x, batch_y, epsilon=8/255)
    X_fgsm.append(fgsm_batch.cpu())
    
    pgd_batch = pgd_attack(model, batch_x, batch_y, epsilon=8/255, alpha=2/255, num_iter=40)
    X_pgd.append(pgd_batch.cpu())

X_fgsm = torch.cat(X_fgsm, dim=0)
X_pgd = torch.cat(X_pgd, dim=0)
print(f"FGSM examples: {X_fgsm.shape}, PGD examples: {X_pgd.shape}")

In [ ]:
# Evaluate three defense configurations
print("Evaluating defense configurations...\n")

results = {}

# Configuration 1: No defense
acc_clean_1 = evaluate_defense(model, X_test, y_test, None)
acc_fgsm_1 = evaluate_defense(model, X_fgsm, y_test, None)
acc_pgd_1 = evaluate_defense(model, X_pgd, y_test, None)
results["No Defense"] = (acc_clean_1, acc_fgsm_1, acc_pgd_1)
print(f"No Defense: Clean={acc_clean_1:.1f}%, FGSM={acc_fgsm_1:.1f}%, PGD={acc_pgd_1:.1f}%")

# Configuration 2: Bit-depth reduction (3-bit) + Median filter (3x3)
def defense_2(x):
    x = bit_depth_reduction(x, bits=3)
    x = median_filter_defense(x, kernel_size=3)
    return x

acc_clean_2 = evaluate_defense(model, X_test, y_test, defense_2)
acc_fgsm_2 = evaluate_defense(model, X_fgsm, y_test, defense_2)
acc_pgd_2 = evaluate_defense(model, X_pgd, y_test, defense_2)
results["Bit-Depth + Median"] = (acc_clean_2, acc_fgsm_2, acc_pgd_2)
print(f"Bit+Median: Clean={acc_clean_2:.1f}%, FGSM={acc_fgsm_2:.1f}%, PGD={acc_pgd_2:.1f}%")

# Configuration 3: Full pipeline (3-bit + Median + JPEG)
def defense_3(x):
    x = bit_depth_reduction(x, bits=3)
    x = median_filter_defense(x, kernel_size=3)
    x = jpeg_compression_defense(x, quality=75)
    return x

acc_clean_3 = evaluate_defense(model, X_test, y_test, defense_3)
acc_fgsm_3 = evaluate_defense(model, X_fgsm, y_test, defense_3)
acc_pgd_3 = evaluate_defense(model, X_pgd, y_test, defense_3)
results["Full Pipeline"] = (acc_clean_3, acc_fgsm_3, acc_pgd_3)
print(f"Full Pipeline: Clean={acc_clean_3:.1f}%, FGSM={acc_fgsm_3:.1f}%, PGD={acc_pgd_3:.1f}%")

In [ ]:
# Display results as a 3x3 table
import pandas as pd

table_data = {
    "Defense Configuration": ["No Defense", "Bit-Depth + Median", "Full Pipeline"],
    "Clean Accuracy (%)": [acc_clean_1, acc_clean_2, acc_clean_3],
    "FGSM Accuracy (%)": [acc_fgsm_1, acc_fgsm_2, acc_fgsm_3],
    "PGD Accuracy (%)": [acc_pgd_1, acc_pgd_2, acc_pgd_3],
}

df = pd.DataFrame(table_data)
print("\n" + "="*90)
print("Defense Evaluation Results")
print("="*90)
print(df.to_string(index=False))
print("="*90)

print(f"\nClean accuracy drop (Full vs. No Defense): {acc_clean_1 - acc_clean_3:.1f} percentage points")
print(f"FGSM robustness gain (Full vs. No Defense): {acc_fgsm_3 - acc_fgsm_1:.1f} percentage points")
print(f"PGD robustness gain (Full vs. No Defense): {acc_pgd_3 - acc_pgd_1:.1f} percentage points")

## Questions

1. Explain why the full defense pipeline shows lower clean accuracy than the no-defense baseline. Which stage of the pipeline (bit-depth, median, or JPEG) do you think contributes most to clean accuracy loss?

2. According to the lesson lecture, JPEG compression removes high-frequency adversarial perturbations. Why might the full pipeline defense be less effective against PGD than against FGSM? (Hint: consider where PGD perturbations concentrate and how multiple pre-processing steps interact.)

In [ ]:
# Verification: confirm all three defense configurations executed successfully
assert len(results) == 3, "Should have 3 defense configurations"
assert "No Defense" in results, "No Defense results missing"
assert "Bit-Depth + Median" in results, "Bit-Depth + Median results missing"
assert "Full Pipeline" in results, "Full Pipeline results missing"

# Check baseline accuracy is reasonable (> 50%)
assert results["No Defense"][0] > 50, f"No defense clean accuracy should be > 50%"

# Check adversarial accuracy is lower than clean for all configurations
for config_name, (clean, fgsm, pgd) in results.items():
    assert fgsm <= clean, f"{config_name}: FGSM accuracy should be <= clean accuracy"
    assert pgd <= clean, f"{config_name}: PGD accuracy should be <= clean accuracy"

print("✓ All verification checks passed!")
print("✓ Defense configurations were implemented correctly.")
print("✓ Adversarial examples were generated and evaluated successfully.")
print("✓ Results are consistent with expected defense behavior.")

## Optional Challenges

1. **Explore adaptive attack (BPDA):** Modify FGSM or PGD to approximate the non-differentiable bit-depth reduction with a smooth function (e.g., sigmoid). Does this increase adversarial accuracy against the full pipeline compared to standard FGSM/PGD?

2. **Tune quality factor:** Test JPEG compression at quality factors Q = 50, Q = 75, Q = 90. Evaluate all three defense configurations with each Q. Plot clean and adversarial accuracy as functions of Q. Where is the sweet spot between clean accuracy and adversarial robustness?

## Completion Criteria

- You trained a ResNet-18 on CIFAR-10 and verified baseline accuracy 70-75% (setup).
- You read the FGSM attack (implemented for you with ART's `FastGradientMethod`) and confirmed perturbations are at epsilon = 8/255 (step 2).
- You implemented the PGD attack with ART's `ProjectedGradientDescent`, alpha = 2/255, 40 iterations, epsilon = 8/255 (step 3).
- You read the bit-depth-reduction defense (implemented for you with ART's `FeatureSqueezing`) and confirmed it quantizes to 3 bits per channel (step 4).
- You implemented a 3x3 median filter defense with ART's `SpatialSmoothingPyTorch` (step 5).
- You implemented the JPEG compression defense at Q = 75 with ART's `JpegCompression` (step 6).
- You evaluated all three defense configurations against clean and adversarial inputs (step 7).
- You displayed results as a 3x3 table: rows are defense variants, columns are clean/FGSM/PGD.
- The verification cell prints 'All verification checks passed!'.